# 09 Cross-Dataset Framework Validation

## Purpose

This notebook consolidates and compares the final results across all three high-stakes decision domains:

- German Credit – Credit Risk
- Pima Diabetes – Healthcare
- COMPAS – Criminal Justice

The objective is to evaluate whether the proposed Explainability-Ethics Score (EES) Framework produces meaningful, transparent, and robust results across different application domains.

## Storyline

Each major section follows the same structure:

```text
Key Finding → Winner Table → Component Breakdown → Full Ranking Table → Cross-Dataset Interpretation
```

This keeps the results chapter readable while preserving traceability into each composite score.

## Research Questions

1. Which models perform best according to baseline predictive performance?
2. Which models produce the strongest SHAP, LIME, and SHAP-LIME agreement evidence?
3. Which models score highest on Fidelity, Stability, Actionability, and Regulatory Alignment?
4. Which models achieve the highest final EES scores?
5. Are EES rankings robust under alternative AHP-derived weighting schemes?
6. Does the EES Framework generalise across credit risk, healthcare, and criminal justice domains?

In [1]:
# =========================================================
# 0. Import Required Libraries
# =========================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 400)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# =========================================================
# 1. Mount Google Drive and Configure Cross-Dataset Paths
# =========================================================

from google.colab import drive

try:
    drive.mount('/content/drive')
except Exception as e:
    print("Drive mount skipped or already available:", e)

ROOT_PATH = Path("/content/drive/MyDrive/Master_Research")
RESULTS_ROOT = ROOT_PATH / "07_Results"

CROSS_DATASET_FOLDER = "09_cross_dataset_framework_validation"
CROSS_DATASET_RESULTS_PATH = RESULTS_ROOT / CROSS_DATASET_FOLDER
TABLES_PATH = CROSS_DATASET_RESULTS_PATH / "tables"
DATASETS_PATH = CROSS_DATASET_RESULTS_PATH / "datasets"
LOGS_PATH = CROSS_DATASET_RESULTS_PATH / "logs"

for path in [CROSS_DATASET_RESULTS_PATH, TABLES_PATH, DATASETS_PATH, LOGS_PATH]:
    path.mkdir(parents=True, exist_ok=True)

DATASET_CONFIGS = {
    "german_credit": {
        "dataset_name": "german_credit",
        "dataset_label": "German Credit",
        "domain": "Credit Risk",
        "results_folder": "german_credit",
        "baseline_folder": "03_baseline_modeling_german_credit",
        "shap_folder": "04_shap_interpretation_german_credit",
        "lime_folder": "05_lime_interpretation_german_credit",
        "disagreement_folder": "06_shap_lime_explaination_disagreement_german_credit",
        "counterfactual_folder": "07_counterfactual_fairness_dice_and_recourse_equity_german_credit",
        "fidelity_folder": "08_02_ees_framework_fidelity_score_german_credit",
        "stability_folder": "08_03_ees_framework_stability_score_german_credit",
        "actionability_folder": "08_04_ees_framework_actionability_score_german_credit",
        "regulatory_folder": "08_05_ees_framework_regulatory_alignment_score_german_credit",
        "ahp_folder": "08_06_ees_framework_ees_ahp_score_german_credit",
        "consolidation_folder": "08_07_results_consolidation_german_credit"
    },
    "pima_diabetes": {
        "dataset_name": "pima_diabetes",
        "dataset_label": "Pima Diabetes",
        "domain": "Healthcare",
        "results_folder": "pima_diabetes",
        "baseline_folder": "03_baseline_modeling_pima_diabetes",
        "shap_folder": "04_shap_interpretation_pima_diabetes",
        "lime_folder": "05_lime_interpretation_pima_diabetes",
        "disagreement_folder": "06_shap_lime_explaination_disagreement_pima_diabetes",
        "counterfactual_folder": "07_counterfactual_fairness_dice_and_recourse_equity_pima_diabetes",
        "fidelity_folder": "08_02_ees_framework_fidelity_score_pima_diabetes",
        "stability_folder": "08_03_ees_framework_stability_score_pima_diabetes",
        "actionability_folder": "08_04_ees_framework_actionability_score_pima_diabetes",
        "regulatory_folder": "08_05_ees_framework_regulatory_alignment_score_pima_diabetes",
        "ahp_folder": "08_06_ees_framework_ees_ahp_score_pima_diabetes",
        "consolidation_folder": "08_07_results_consolidation_pima_diabetes"
    },
    "compas": {
        "dataset_name": "compas",
        "dataset_label": "COMPAS",
        "domain": "Criminal Justice",
        "results_folder": "compas",
        "baseline_folder": "03_baseline_modeling_compas",
        "shap_folder": "04_shap_interpretation_compas",
        "lime_folder": "05_lime_interpretation_compas",
        "disagreement_folder": "06_shap_lime_explaination_disagreement_compas",
        "counterfactual_folder": "07_counterfactual_fairness_dice_and_recourse_equity_compas",
        "fidelity_folder": "08_02_ees_framework_fidelity_score_compas",
        "stability_folder": "08_03_ees_framework_stability_score_compas",
        "actionability_folder": "08_04_ees_framework_actionability_score_compas",
        "regulatory_folder": "08_05_ees_framework_regulatory_alignment_score_compas",
        "ahp_folder": "08_06_ees_framework_ees_ahp_score_compas",
        "consolidation_folder": "08_07_results_consolidation_compas"
    }
}

print("Cross-dataset output path:", CROSS_DATASET_RESULTS_PATH)

Mounted at /content/drive
Cross-dataset output path: /content/drive/MyDrive/Master_Research/07_Results/09_cross_dataset_framework_validation


In [3]:
# =========================================================
# 2. Helper Functions
# =========================================================

def read_csv_safely(path, label, required=True):
    path = Path(path)
    if not path.exists():
        if required:
            raise FileNotFoundError(f"{label} not found at: {path}")
        print(f"Optional file not found: {label} at {path}")
        return pd.DataFrame()
    return pd.read_csv(path)


def find_first_existing_file(candidate_paths, label, required=True):
    for path in candidate_paths:
        path = Path(path)
        if path.exists():
            print(f"{label} found at: {path}")
            return path
    if required:
        checked = "\n".join(str(path) for path in candidate_paths)
        raise FileNotFoundError(f"{label} not found. Checked:\n{checked}")
    print(f"{label} not found.")
    return None


def save_df(df, filename, folder=None, index=False):
    if folder is None:
        folder = TABLES_PATH
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    output_path = folder / filename
    df.to_csv(output_path, index=index)
    print(f"Saved: {output_path}")
    return output_path


def save_excel(dfs, filename, folder=None):
    if folder is None:
        folder = TABLES_PATH
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    output_path = folder / filename
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for sheet_name, df in dfs.items():
            if isinstance(df, pd.DataFrame):
                df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
    print(f"Saved: {output_path}")
    return output_path


def add_dataset_columns(df, config):
    if df.empty:
        return df
    out = df.copy()
    if "dataset_name" not in out.columns:
        out.insert(0, "dataset_name", config["dataset_name"])
    if "dataset_label" not in out.columns:
        out.insert(1, "dataset_label", config["dataset_label"])
    if "domain" not in out.columns:
        out.insert(2, "domain", config["domain"])
    return out


def rank_within_dataset(df, score_col, rank_col, ascending=False):
    out = df.copy()
    out[rank_col] = (
        out.groupby("dataset_name")[score_col]
        .rank(ascending=ascending, method="dense")
        .astype(int)
    )
    return out


def get_dataset_winners(df, score_col, rank_col, value_name):
    if df.empty:
        return pd.DataFrame()
    temp = df.copy()
    if rank_col not in temp.columns:
        temp = rank_within_dataset(temp, score_col, rank_col, ascending=False)
    winners = (
        temp[temp[rank_col] == 1]
        [["dataset_name", "dataset_label", "domain", "model_id", "model_name", score_col]]
        .rename(columns={"model_id": "winner_model_id", "model_name": "winner_model_name", score_col: value_name})
        .sort_values("dataset_name")
        .reset_index(drop=True)
    )
    return winners


def safe_cols(df, cols):
    return [col for col in cols if col in df.columns]

# 3. Load Results from All Three Dataset Pipelines

This section loads the required outputs from each completed dataset pipeline.

In [4]:
# =========================================================
# 3. Load Results from All Three Dataset Pipelines
# =========================================================

def get_paths(dataset_root, folder, file_candidates):
    base = dataset_root / folder
    paths = []
    for file_name in file_candidates:
        paths.append(base / "datasets" / file_name)
        paths.append(base / "tables" / file_name)
    return paths

OBJECT_SPECS = {
    "baseline": ("baseline_folder", ["baseline_model_performance.csv", "model_performance.csv"]),
    "shap_long": ("shap_folder", ["all_test_local_shap_long.csv", "shap_all_test_long.csv"]),
    "lime_fidelity": ("lime_folder", ["lime_fidelity_inputs.csv"]),
    "agreement": ("disagreement_folder", ["shap_lime_agreement_for_fidelity.csv", "model_level_shap_lime_agreement_summary.csv"]),
    "counterfactual_actionability": ("counterfactual_folder", ["ees_actionability_model_scores.csv", "actionability_score_summary.csv"]),
    "fidelity": ("fidelity_folder", ["ees_fidelity_model_scores.csv", "fidelity_score_summary.csv"]),
    "stability": ("stability_folder", ["ees_stability_model_scores.csv", "stability_score_summary.csv"]),
    "actionability": ("actionability_folder", ["ees_actionability_model_scores.csv", "actionability_score_summary.csv"]),
    "regulatory": ("regulatory_folder", ["ees_regulatory_alignment_model_scores.csv", "regulatory_alignment_score_summary.csv"]),
    "ees": ("ahp_folder", ["ees_final_model_scores.csv", "ees_ahp_score_summary.csv"]),
    "consolidated": ("consolidation_folder", ["final_consolidated_results.csv"]),
    "publication": ("consolidation_folder", ["publication_ready_summary.csv"]),
    "ranking_robustness": ("consolidation_folder", ["ees_ranking_robustness_summary.csv"]),
    "ahp_diagnostics": ("consolidation_folder", ["ahp_diagnostics_summary.csv", "ees_ahp_scheme_consistency_diagnostics.csv"]),
    "ahp_scheme_weights": ("consolidation_folder", ["ahp_scheme_weights_summary.csv", "ees_ahp_scheme_weights.csv"]),
    "ahp_sensitivity": ("consolidation_folder", ["ees_sensitivity_summary.csv", "ees_weighting_sensitivity_long.csv", "ees_weighting_sensitivity.csv"]),
}

loaded = {key: [] for key in OBJECT_SPECS}
load_registry_rows = []

for dataset_key, config in DATASET_CONFIGS.items():
    dataset_root = RESULTS_ROOT / config["results_folder"]
    for object_name, (folder_key, file_candidates) in OBJECT_SPECS.items():
        paths = get_paths(dataset_root, config[folder_key], file_candidates)

        # fallback for AHP outputs directly in 08_06 if consolidation does not contain them
        if object_name in ["ranking_robustness", "ahp_diagnostics", "ahp_scheme_weights", "ahp_sensitivity"]:
            paths += get_paths(dataset_root, config["ahp_folder"], file_candidates)

        found_path = find_first_existing_file(paths, f"{config['dataset_label']} - {object_name}", required=False)
        load_registry_rows.append({
            "dataset_name": config["dataset_name"],
            "dataset_label": config["dataset_label"],
            "domain": config["domain"],
            "object_name": object_name,
            "file_found": found_path is not None,
            "file_path": str(found_path) if found_path else None
        })
        if found_path:
            loaded[object_name].append(add_dataset_columns(read_csv_safely(found_path, object_name, required=False), config))
        else:
            loaded[object_name].append(pd.DataFrame())

load_registry_df = pd.DataFrame(load_registry_rows)

baseline_all_df = pd.concat(loaded["baseline"], ignore_index=True)
shap_long_all_df = pd.concat(loaded["shap_long"], ignore_index=True)
lime_fidelity_all_df = pd.concat(loaded["lime_fidelity"], ignore_index=True)
agreement_all_df = pd.concat(loaded["agreement"], ignore_index=True)
counterfactual_actionability_all_df = pd.concat(loaded["counterfactual_actionability"], ignore_index=True)
fidelity_all_df = pd.concat(loaded["fidelity"], ignore_index=True)
stability_all_df = pd.concat(loaded["stability"], ignore_index=True)
actionability_all_df = pd.concat(loaded["actionability"], ignore_index=True)
regulatory_all_df = pd.concat(loaded["regulatory"], ignore_index=True)
ees_all_df = pd.concat(loaded["ees"], ignore_index=True)
consolidated_all_df = pd.concat(loaded["consolidated"], ignore_index=True)
publication_all_df = pd.concat(loaded["publication"], ignore_index=True)
ranking_robustness_all_df = pd.concat(loaded["ranking_robustness"], ignore_index=True)
ahp_diagnostics_all_df = pd.concat(loaded["ahp_diagnostics"], ignore_index=True)
ahp_scheme_weights_all_df = pd.concat(loaded["ahp_scheme_weights"], ignore_index=True)
ahp_sensitivity_all_df = pd.concat(loaded["ahp_sensitivity"], ignore_index=True)

display(load_registry_df)
for name, df in [
    ("baseline", baseline_all_df), ("shap_long", shap_long_all_df), ("lime_fidelity", lime_fidelity_all_df),
    ("agreement", agreement_all_df), ("fidelity", fidelity_all_df), ("stability", stability_all_df),
    ("actionability", actionability_all_df), ("regulatory", regulatory_all_df), ("ees", ees_all_df),
    ("ranking_robustness", ranking_robustness_all_df), ("ahp_diagnostics", ahp_diagnostics_all_df)
]:
    print(f"{name}: {df.shape}")

German Credit - baseline found at: /content/drive/MyDrive/Master_Research/07_Results/german_credit/03_baseline_modeling_german_credit/datasets/baseline_model_performance.csv
German Credit - shap_long found at: /content/drive/MyDrive/Master_Research/07_Results/german_credit/04_shap_interpretation_german_credit/datasets/all_test_local_shap_long.csv
German Credit - lime_fidelity found at: /content/drive/MyDrive/Master_Research/07_Results/german_credit/05_lime_interpretation_german_credit/datasets/lime_fidelity_inputs.csv
German Credit - agreement found at: /content/drive/MyDrive/Master_Research/07_Results/german_credit/06_shap_lime_explaination_disagreement_german_credit/datasets/shap_lime_agreement_for_fidelity.csv
German Credit - counterfactual_actionability found at: /content/drive/MyDrive/Master_Research/07_Results/german_credit/07_counterfactual_fairness_dice_and_recourse_equity_german_credit/datasets/ees_actionability_model_scores.csv
German Credit - fidelity found at: /content/driv

,dataset_name,dataset_label,domain,object_name,file_found,file_path
0,german_credit,German Credit,Credit Risk,baseline,True,/content/drive/MyDrive/Master_Research/07_Resu...
1,german_credit,German Credit,Credit Risk,shap_long,True,/content/drive/MyDrive/Master_Research/07_Resu...
2,german_credit,German Credit,Credit Risk,lime_fidelity,True,/content/drive/MyDrive/Master_Research/07_Resu...
3,german_credit,German Credit,Credit Risk,agreement,True,/content/drive/MyDrive/Master_Research/07_Resu...
4,german_credit,German Credit,Credit Risk,counterfactual_actionability,True,/content/drive/MyDrive/Master_Research/07_Resu...
5,german_credit,German Credit,Credit Risk,fidelity,True,/content/drive/MyDrive/Master_Research/07_Resu...
6,german_credit,German Credit,Credit Risk,stability,True,/content/drive/MyDrive/Master_Research/07_Resu...
7,german_credit,German Credit,Credit Risk,actionability,True,/content/drive/MyDrive/Master_Research/07_Resu...
8,german_credit,German Credit,Credit Risk,regulatory,True,/content/drive/MyDrive/Master_Research/07_Resu...
9,german_credit,German Credit,Credit Risk,ees,True,/content/drive/MyDrive/Master_Research/07_Resu...


baseline: (15, 21)
shap_long: (560160, 11)
lime_fidelity: (19270, 13)
agreement: (15, 12)
fidelity: (15, 19)
stability: (15, 18)
actionability: (15, 21)
regulatory: (15, 15)
ees: (15, 36)
ranking_robustness: (15, 16)
ahp_diagnostics: (18, 11)


# 4. Dataset Overview

## Objective

Provide high-level context on the three domains used to evaluate the EES Framework.

In [5]:
# =========================================================
# 4. Dataset Overview
# =========================================================

dataset_overview_df = pd.DataFrame([
    {
        "dataset_name": config["dataset_name"],
        "dataset_label": config["dataset_label"],
        "domain": config["domain"],
        "pipeline_status": "Loaded" if load_registry_df[(load_registry_df["dataset_name"] == config["dataset_name"]) & (load_registry_df["file_found"])].shape[0] > 0 else "Not loaded"
    }
    for config in DATASET_CONFIGS.values()
])

if not consolidated_all_df.empty and "model_id" in consolidated_all_df.columns:
    model_count_df = consolidated_all_df.groupby(["dataset_name", "dataset_label"], as_index=False).agg(n_models=("model_id", "nunique"))
    dataset_overview_df = dataset_overview_df.merge(model_count_df, on=["dataset_name", "dataset_label"], how="left")

display(dataset_overview_df)

,dataset_name,dataset_label,domain,pipeline_status,n_models
0,german_credit,German Credit,Credit Risk,Loaded,5
1,pima_diabetes,Pima Diabetes,Healthcare,Loaded,5
2,compas,COMPAS,Criminal Justice,Loaded,5


# 5. Baseline Model Performance Comparison

## Objective

Evaluate predictive model performance before considering explainability and ethics.

## Storyline Example

The Business Score is decomposed into Recall, ROC-AUC, and Precision so that the reader can see why a champion model wins.

In [6]:
# =========================================================
# 5. Baseline Model Performance Comparison
# =========================================================

baseline_component_comparison_df = baseline_all_df[safe_cols(baseline_all_df, [
    "dataset_name", "dataset_label", "domain", "model_id", "model_name", "recall", "roc_auc", "precision",
    "accuracy", "f1_score", "average_precision", "business_priority_score"
])].copy()

if not baseline_component_comparison_df.empty and "business_priority_score" in baseline_component_comparison_df.columns:
    baseline_component_comparison_df = rank_within_dataset(baseline_component_comparison_df, "business_priority_score", "business_priority_rank", ascending=False)

baseline_winner_df = get_dataset_winners(baseline_component_comparison_df, "business_priority_score", "business_priority_rank", "business_priority_score") if not baseline_component_comparison_df.empty else pd.DataFrame()
baseline_full_ranking_df = baseline_component_comparison_df.sort_values(["dataset_name", "business_priority_rank", "model_id"]).reset_index(drop=True) if not baseline_component_comparison_df.empty else pd.DataFrame()

display(baseline_winner_df)
display(baseline_component_comparison_df)
display(baseline_full_ranking_df)

,dataset_name,dataset_label,domain,winner_model_id,winner_model_name,business_priority_score
0,compas,COMPAS,Criminal Justice,random_forest,Random Forest,0.854279
1,german_credit,German Credit,Credit Risk,xgboost,XGBoost,0.619293
2,pima_diabetes,Pima Diabetes,Healthcare,decision_tree,Decision Tree,0.717647


,dataset_name,dataset_label,domain,model_id,model_name,recall,roc_auc,precision,accuracy,f1_score,average_precision,business_priority_score,business_priority_rank
0,german_credit,German Credit,Credit Risk,xgboost,XGBoost,0.516667,0.781429,0.632653,0.765000,0.568807,0.656290,0.619293,1
1,german_credit,German Credit,Credit Risk,decision_tree,Decision Tree,0.533333,0.739940,0.470588,0.680000,0.500000,0.551867,0.582766,2
2,german_credit,German Credit,Credit Risk,mlp,MLP,0.483333,0.740238,0.517857,0.710000,0.500000,0.552379,0.567310,3
3,german_credit,German Credit,Credit Risk,random_forest,Random Forest,0.400000,0.769167,0.615385,0.745000,0.484848,0.618334,0.553827,4
4,german_credit,German Credit,Credit Risk,logistic_regression,Logistic Regression,0.383333,0.761429,0.589744,0.735000,0.464646,0.617207,0.538044,5
5,pima_diabetes,Pima Diabetes,Healthcare,decision_tree,Decision Tree,0.722222,0.762222,0.639344,0.759740,0.678261,0.585402,0.717647,1
6,pima_diabetes,Pima Diabetes,Healthcare,xgboost,XGBoost,0.648148,0.826481,0.686275,0.772727,0.666667,0.671049,0.709273,2
7,pima_diabetes,Pima Diabetes,Healthcare,random_forest,Random Forest,0.537037,0.815093,0.659091,0.740260,0.591837,0.698897,0.644864,3
8,pima_diabetes,Pima Diabetes,Healthcare,mlp,MLP,0.500000,0.814630,0.613636,0.714286,0.551020,0.689200,0.617116,4
9,pima_diabetes,Pima Diabetes,Healthcare,logistic_regression,Logistic Regression,0.500000,0.812222,0.586957,0.701299,0.540000,0.673052,0.611058,5


,dataset_name,dataset_label,domain,model_id,model_name,recall,roc_auc,precision,accuracy,f1_score,average_precision,business_priority_score,business_priority_rank
0,compas,COMPAS,Criminal Justice,random_forest,Random Forest,0.834917,0.903823,0.807582,0.824857,0.821022,0.901771,0.854279,1
1,compas,COMPAS,Criminal Justice,xgboost,XGBoost,0.675178,0.760197,0.680431,0.691143,0.677794,0.735896,0.710761,2
2,compas,COMPAS,Criminal Justice,mlp,MLP,0.667458,0.739305,0.661955,0.676000,0.664695,0.693323,0.694546,3
3,compas,COMPAS,Criminal Justice,decision_tree,Decision Tree,0.682304,0.723393,0.655822,0.674857,0.668801,0.673207,0.690795,4
4,compas,COMPAS,Criminal Justice,logistic_regression,Logistic Regression,0.633610,0.726255,0.670226,0.673714,0.651404,0.686058,0.681653,5
5,german_credit,German Credit,Credit Risk,xgboost,XGBoost,0.516667,0.781429,0.632653,0.765000,0.568807,0.656290,0.619293,1
6,german_credit,German Credit,Credit Risk,decision_tree,Decision Tree,0.533333,0.739940,0.470588,0.680000,0.500000,0.551867,0.582766,2
7,german_credit,German Credit,Credit Risk,mlp,MLP,0.483333,0.740238,0.517857,0.710000,0.500000,0.552379,0.567310,3
8,german_credit,German Credit,Credit Risk,random_forest,Random Forest,0.400000,0.769167,0.615385,0.745000,0.484848,0.618334,0.553827,4
9,german_credit,German Credit,Credit Risk,logistic_regression,Logistic Regression,0.383333,0.761429,0.589744,0.735000,0.464646,0.617207,0.538044,5


# 6. SHAP Global Explanation Comparison

## Objective

Identify Top 5 most influential SHAP features by dataset.

In [7]:
# =========================================================
# 6. SHAP Global Explanation Comparison - Top 5 Features
# =========================================================

if shap_long_all_df.empty:
    shap_feature_importance_all_df = pd.DataFrame()
    shap_top_feature_by_dataset_df = pd.DataFrame()
    shap_top5_features_by_dataset_df = pd.DataFrame()
else:
    shap_value_col = next((c for c in ["abs_shap_value", "mean_abs_shap_value", "shap_importance"] if c in shap_long_all_df.columns), None)
    if shap_value_col is None or "feature" not in shap_long_all_df.columns:
        shap_feature_importance_all_df = pd.DataFrame()
        shap_top_feature_by_dataset_df = pd.DataFrame()
        shap_top5_features_by_dataset_df = pd.DataFrame()
    else:
        shap_feature_importance_all_df = shap_long_all_df.groupby(["dataset_name", "dataset_label", "domain", "feature"], as_index=False).agg(mean_abs_shap_value=(shap_value_col, "mean"))
        shap_feature_importance_all_df["feature_rank"] = shap_feature_importance_all_df.groupby("dataset_name")["mean_abs_shap_value"].rank(ascending=False, method="dense").astype(int)
        shap_top5_features_by_dataset_df = shap_feature_importance_all_df[shap_feature_importance_all_df["feature_rank"] <= 5].sort_values(["dataset_name", "feature_rank"]).reset_index(drop=True)
        shap_top_feature_by_dataset_df = shap_feature_importance_all_df[shap_feature_importance_all_df["feature_rank"] == 1].sort_values("dataset_name").reset_index(drop=True)

display(shap_top_feature_by_dataset_df)
display(shap_top5_features_by_dataset_df)

,dataset_name,dataset_label,domain,feature,mean_abs_shap_value,feature_rank
0,compas,COMPAS,Criminal Justice,priors_count,0.186183,1
1,german_credit,German Credit,Credit Risk,Checking account_Missing,0.159608,1
2,pima_diabetes,Pima Diabetes,Healthcare,Glucose,0.320497,1


,dataset_name,dataset_label,domain,feature,mean_abs_shap_value,feature_rank
0,compas,COMPAS,Criminal Justice,priors_count,0.186183,1
1,compas,COMPAS,Criminal Justice,age,0.137747,2
2,compas,COMPAS,Criminal Justice,c_days_from_compas,0.048672,3
3,compas,COMPAS,Criminal Justice,days_b_screening_arrest,0.038701,4
4,compas,COMPAS,Criminal Justice,sex_Female,0.025909,5
5,german_credit,German Credit,Credit Risk,Checking account_Missing,0.159608,1
6,german_credit,German Credit,Credit Risk,Duration,0.143605,2
7,german_credit,German Credit,Credit Risk,Checking account_little,0.084512,3
8,german_credit,German Credit,Credit Risk,Saving accounts_little,0.073556,4
9,german_credit,German Credit,Credit Risk,Credit amount,0.054389,5


# 7. LIME Local Fidelity Comparison

## Objective

Compare LIME local surrogate fidelity across datasets and models.

In [8]:
# =========================================================
# 7. LIME Local Fidelity Comparison
# =========================================================

if lime_fidelity_all_df.empty:
    lime_ranking_df = pd.DataFrame(); lime_winner_df = pd.DataFrame()
else:
    lime_score_col = next((c for c in ["mean_lime_r2_score", "lime_r2_score", "lime_fidelity_score"] if c in lime_fidelity_all_df.columns), None)
    if lime_score_col is None:
        lime_ranking_df = pd.DataFrame(); lime_winner_df = pd.DataFrame()
    else:
        if "test_instance_position" in lime_fidelity_all_df.columns:
            lime_ranking_df = lime_fidelity_all_df.groupby(["dataset_name", "dataset_label", "domain", "model_id", "model_name"], as_index=False).agg(mean_lime_r2_score=(lime_score_col, "mean"))
        else:
            lime_ranking_df = lime_fidelity_all_df.copy()
            if lime_score_col != "mean_lime_r2_score": lime_ranking_df = lime_ranking_df.rename(columns={lime_score_col: "mean_lime_r2_score"})
        lime_ranking_df = rank_within_dataset(lime_ranking_df, "mean_lime_r2_score", "lime_rank", ascending=False).sort_values(["dataset_name", "lime_rank", "model_id"]).reset_index(drop=True)
        lime_winner_df = get_dataset_winners(lime_ranking_df, "mean_lime_r2_score", "lime_rank", "mean_lime_r2_score")

display(lime_winner_df)
display(lime_ranking_df)

,dataset_name,dataset_label,domain,winner_model_id,winner_model_name,mean_lime_r2_score
0,compas,COMPAS,Criminal Justice,xgboost,XGBoost,0.518743
1,german_credit,German Credit,Credit Risk,logistic_regression,Logistic Regression,0.663835
2,pima_diabetes,Pima Diabetes,Healthcare,logistic_regression,Logistic Regression,0.317475


,dataset_name,dataset_label,domain,model_id,model_name,mean_lime_r2_score,lime_rank
0,compas,COMPAS,Criminal Justice,xgboost,XGBoost,0.518743,1
1,compas,COMPAS,Criminal Justice,mlp,MLP,0.456811,2
2,compas,COMPAS,Criminal Justice,random_forest,Random Forest,0.404200,3
3,compas,COMPAS,Criminal Justice,logistic_regression,Logistic Regression,0.366766,4
4,compas,COMPAS,Criminal Justice,decision_tree,Decision Tree,0.263802,5
5,german_credit,German Credit,Credit Risk,logistic_regression,Logistic Regression,0.663835,1
6,german_credit,German Credit,Credit Risk,random_forest,Random Forest,0.514927,2
7,german_credit,German Credit,Credit Risk,xgboost,XGBoost,0.396174,3
8,german_credit,German Credit,Credit Risk,decision_tree,Decision Tree,0.333585,4
9,german_credit,German Credit,Credit Risk,mlp,MLP,0.171749,5


# 8. SHAP-LIME Agreement Comparison

## Objective

Assess explanation consistency between SHAP and LIME.

In [9]:
# =========================================================
# 8. SHAP-LIME Agreement Comparison
# =========================================================

if agreement_all_df.empty:
    agreement_ranking_df = pd.DataFrame(); agreement_winner_df = pd.DataFrame()
else:
    agreement_score_col = next((c for c in ["shap_lime_agreement_score", "mean_agreement_score", "agreement_score", "normalised_kendall_tau", "normalized_kendall_tau"] if c in agreement_all_df.columns), None)
    if agreement_score_col is None:
        agreement_ranking_df = pd.DataFrame(); agreement_winner_df = pd.DataFrame()
    else:
        agreement_ranking_df = agreement_all_df.copy()
        if agreement_score_col != "shap_lime_agreement_score": agreement_ranking_df = agreement_ranking_df.rename(columns={agreement_score_col: "shap_lime_agreement_score"})
        if "test_instance_position" in agreement_ranking_df.columns:
            agreement_ranking_df = agreement_ranking_df.groupby(["dataset_name", "dataset_label", "domain", "model_id", "model_name"], as_index=False).agg(shap_lime_agreement_score=("shap_lime_agreement_score", "mean"))
        agreement_ranking_df = rank_within_dataset(agreement_ranking_df, "shap_lime_agreement_score", "agreement_rank", ascending=False).sort_values(["dataset_name", "agreement_rank", "model_id"]).reset_index(drop=True)
        agreement_winner_df = get_dataset_winners(agreement_ranking_df, "shap_lime_agreement_score", "agreement_rank", "shap_lime_agreement_score")

display(agreement_winner_df)
display(agreement_ranking_df)

,dataset_name,dataset_label,domain,winner_model_id,winner_model_name,shap_lime_agreement_score
0,compas,COMPAS,Criminal Justice,decision_tree,Decision Tree,0.511505
1,german_credit,German Credit,Credit Risk,random_forest,Random Forest,0.588673
2,pima_diabetes,Pima Diabetes,Healthcare,logistic_regression,Logistic Regression,0.867347


,dataset_name,dataset_label,domain,model_id,model_name,n_instances,mean_kendall_tau,shap_lime_agreement_score,shap_lime_agreement_score_pct,shap_lime_agreement_rank,shap_lime_agreement_band,mean_top_k_overlap_ratio,agreement_rank
0,compas,COMPAS,Criminal Justice,decision_tree,Decision Tree,3500,0.023010,0.511505,51.150489,1,Weak agreement,0.448543,1
1,compas,COMPAS,Criminal Justice,random_forest,Random Forest,3500,-0.121791,0.439104,43.910428,2,Weak agreement,0.383429,2
2,compas,COMPAS,Criminal Justice,xgboost,XGBoost,3500,-0.129536,0.435232,43.523177,3,Weak agreement,0.428143,3
3,compas,COMPAS,Criminal Justice,mlp,MLP,3500,-0.309864,0.345068,34.506810,4,Poor agreement,0.288286,4
4,compas,COMPAS,Criminal Justice,logistic_regression,Logistic Regression,3500,-0.326682,0.336659,33.665877,5,Poor agreement,0.341257,5
5,german_credit,German Credit,Credit Risk,random_forest,Random Forest,200,0.177347,0.588673,58.867329,1,Moderate agreement,0.604500,1
6,german_credit,German Credit,Credit Risk,decision_tree,Decision Tree,200,0.149506,0.574753,57.475288,2,Moderate agreement,0.745500,2
7,german_credit,German Credit,Credit Risk,xgboost,XGBoost,200,0.102219,0.551109,55.110948,3,Moderate agreement,0.668000,3
8,german_credit,German Credit,Credit Risk,logistic_regression,Logistic Regression,200,0.073769,0.536884,53.688425,4,Weak agreement,0.515500,4
9,german_credit,German Credit,Credit Risk,mlp,MLP,200,-0.304491,0.347755,34.775466,5,Poor agreement,0.386500,5


# 9. Counterfactual Recourse Comparison

## Objective

Compare recourse availability, proximity, and sparsity across datasets.

## Storyline Example

The winner table identifies the strongest model per dataset, while the component breakdown explains why it won.

In [10]:
# =========================================================
# 9. Counterfactual Recourse Comparison
# =========================================================

base_cols = ["dataset_name", "dataset_label", "domain", "model_id", "model_name"]
counterfactual_component_df = counterfactual_actionability_all_df[safe_cols(counterfactual_actionability_all_df, base_cols + ['availability_score', 'proximity_score', 'sparsity_score', 'actionability_score', 'n_unfavourable_instances', 'n_actionable_recourse_instances', 'mean_min_normalized_numeric_distance', 'mean_changed_features'])].copy()

if not counterfactual_component_df.empty and "counterfactual_actionability_rank" not in counterfactual_component_df.columns and "actionability_score" in counterfactual_component_df.columns:
    counterfactual_component_df = rank_within_dataset(counterfactual_component_df, "actionability_score", "counterfactual_actionability_rank", ascending=False)

counterfactual_winner_df = get_dataset_winners(counterfactual_component_df, "actionability_score", "counterfactual_actionability_rank", "actionability_score") if not counterfactual_component_df.empty and "actionability_score" in counterfactual_component_df.columns else pd.DataFrame()

counterfactual_ranking_df = (
    counterfactual_component_df
    .sort_values(["dataset_name", "counterfactual_actionability_rank", "model_id"])
    .reset_index(drop=True)
) if not counterfactual_component_df.empty and "counterfactual_actionability_rank" in counterfactual_component_df.columns else pd.DataFrame()

display(counterfactual_winner_df)
display(counterfactual_component_df)
display(counterfactual_ranking_df)

,dataset_name,dataset_label,domain,winner_model_id,winner_model_name,actionability_score
0,compas,COMPAS,Criminal Justice,decision_tree,Decision Tree,0.950537
1,german_credit,German Credit,Credit Risk,decision_tree,Decision Tree,0.940371
2,pima_diabetes,Pima Diabetes,Healthcare,decision_tree,Decision Tree,0.942250


,dataset_name,dataset_label,domain,model_id,model_name,availability_score,proximity_score,sparsity_score,actionability_score,n_unfavourable_instances,n_actionable_recourse_instances,mean_min_normalized_numeric_distance,mean_changed_features,counterfactual_actionability_rank
0,german_credit,German Credit,Credit Risk,decision_tree,Decision Tree,1.000000,0.974709,0.846405,0.940371,68,68,0.025291,1.382353,1
1,german_credit,German Credit,Credit Risk,xgboost,XGBoost,1.000000,0.955465,0.859410,0.938292,49,49,0.044535,1.265306,2
2,german_credit,German Credit,Credit Risk,random_forest,Random Forest,1.000000,0.963671,0.831909,0.931860,39,39,0.036329,1.512821,3
3,german_credit,German Credit,Credit Risk,mlp,MLP,1.000000,0.946515,0.847222,0.931246,56,56,0.053485,1.375000,4
4,german_credit,German Credit,Credit Risk,logistic_regression,Logistic Regression,1.000000,0.953052,0.817664,0.923572,39,39,0.046948,1.641026,5
5,pima_diabetes,Pima Diabetes,Healthcare,decision_tree,Decision Tree,1.000000,0.955848,0.870902,0.942250,61,61,0.044152,1.032787,1
6,pima_diabetes,Pima Diabetes,Healthcare,xgboost,XGBoost,1.000000,0.952764,0.862745,0.938503,51,51,0.047236,1.098039,2
7,pima_diabetes,Pima Diabetes,Healthcare,random_forest,Random Forest,1.000000,0.952648,0.855114,0.935921,44,44,0.047352,1.159091,3
8,pima_diabetes,Pima Diabetes,Healthcare,mlp,MLP,1.000000,0.945213,0.857955,0.934389,44,44,0.054787,1.136364,4
9,pima_diabetes,Pima Diabetes,Healthcare,logistic_regression,Logistic Regression,1.000000,0.943100,0.847826,0.930309,46,46,0.056900,1.217391,5


,dataset_name,dataset_label,domain,model_id,model_name,availability_score,proximity_score,sparsity_score,actionability_score,n_unfavourable_instances,n_actionable_recourse_instances,mean_min_normalized_numeric_distance,mean_changed_features,counterfactual_actionability_rank
0,compas,COMPAS,Criminal Justice,decision_tree,Decision Tree,1.000000,0.959375,0.892237,0.950537,1752,1752,0.040625,1.077626,1
1,compas,COMPAS,Criminal Justice,logistic_regression,Logistic Regression,1.000000,0.958794,0.887626,0.948806,1592,1592,0.041206,1.123744,2
2,compas,COMPAS,Criminal Justice,xgboost,XGBoost,1.000000,0.955618,0.882705,0.946108,1671,1671,0.044382,1.172950,3
3,compas,COMPAS,Criminal Justice,mlp,MLP,1.000000,0.950526,0.881861,0.944129,1698,1698,0.049474,1.181390,4
4,compas,COMPAS,Criminal Justice,random_forest,Random Forest,1.000000,0.956268,0.875014,0.943761,1741,1741,0.043732,1.249856,5
5,german_credit,German Credit,Credit Risk,decision_tree,Decision Tree,1.000000,0.974709,0.846405,0.940371,68,68,0.025291,1.382353,1
6,german_credit,German Credit,Credit Risk,xgboost,XGBoost,1.000000,0.955465,0.859410,0.938292,49,49,0.044535,1.265306,2
7,german_credit,German Credit,Credit Risk,random_forest,Random Forest,1.000000,0.963671,0.831909,0.931860,39,39,0.036329,1.512821,3
8,german_credit,German Credit,Credit Risk,mlp,MLP,1.000000,0.946515,0.847222,0.931246,56,56,0.053485,1.375000,4
9,german_credit,German Credit,Credit Risk,logistic_regression,Logistic Regression,1.000000,0.953052,0.817664,0.923572,39,39,0.046948,1.641026,5


# 10. Fidelity Pillar Comparison

## Objective

Compare Fidelity and its three subcomponents.

## Storyline Example

The winner table identifies the strongest model per dataset, while the component breakdown explains why it won.

In [11]:
# =========================================================
# 10. Fidelity Pillar Comparison
# =========================================================

base_cols = ["dataset_name", "dataset_label", "domain", "model_id", "model_name"]
fidelity_component_df = fidelity_all_df[safe_cols(fidelity_all_df, base_cols + ['shap_reconstruction_fidelity_score', 'lime_fidelity_score', 'shap_lime_agreement_score', 'fidelity_score', 'fidelity_rank', 'fidelity_band'])].copy()

if not fidelity_component_df.empty and "fidelity_rank" not in fidelity_component_df.columns and "fidelity_score" in fidelity_component_df.columns:
    fidelity_component_df = rank_within_dataset(fidelity_component_df, "fidelity_score", "fidelity_rank", ascending=False)

fidelity_winner_df = get_dataset_winners(fidelity_component_df, "fidelity_score", "fidelity_rank", "fidelity_score") if not fidelity_component_df.empty and "fidelity_score" in fidelity_component_df.columns else pd.DataFrame()

fidelity_ranking_df = (
    fidelity_component_df
    .sort_values(["dataset_name", "fidelity_rank", "model_id"])
    .reset_index(drop=True)
) if not fidelity_component_df.empty and "fidelity_rank" in fidelity_component_df.columns else pd.DataFrame()

display(fidelity_winner_df)
display(fidelity_component_df)
display(fidelity_ranking_df)

,dataset_name,dataset_label,domain,winner_model_id,winner_model_name,fidelity_score
0,compas,COMPAS,Criminal Justice,xgboost,XGBoost,0.651325
1,german_credit,German Credit,Credit Risk,logistic_regression,Logistic Regression,0.733573
2,pima_diabetes,Pima Diabetes,Healthcare,logistic_regression,Logistic Regression,0.728274


,dataset_name,dataset_label,domain,model_id,model_name,shap_reconstruction_fidelity_score,lime_fidelity_score,shap_lime_agreement_score,fidelity_score,fidelity_rank,fidelity_band
0,german_credit,German Credit,Credit Risk,logistic_regression,Logistic Regression,1.000000,0.663835,0.536884,0.733573,1,Moderate-high fidelity
1,german_credit,German Credit,Credit Risk,random_forest,Random Forest,0.999333,0.514927,0.588673,0.700978,2,Moderate-high fidelity
2,german_credit,German Credit,Credit Risk,xgboost,XGBoost,1.000000,0.396174,0.551109,0.649094,3,Moderate-high fidelity
3,german_credit,German Credit,Credit Risk,decision_tree,Decision Tree,1.000000,0.333585,0.574753,0.636113,4,Moderate-high fidelity
4,german_credit,German Credit,Credit Risk,mlp,MLP,1.000000,0.171749,0.347755,0.506501,5,Moderate fidelity
5,pima_diabetes,Pima Diabetes,Healthcare,logistic_regression,Logistic Regression,1.000000,0.317475,0.867347,0.728274,1,Moderate-high fidelity
6,pima_diabetes,Pima Diabetes,Healthcare,mlp,MLP,1.000000,0.290141,0.794063,0.694735,2,Moderate-high fidelity
7,pima_diabetes,Pima Diabetes,Healthcare,random_forest,Random Forest,0.998148,0.276355,0.766234,0.680246,3,Moderate-high fidelity
8,pima_diabetes,Pima Diabetes,Healthcare,xgboost,XGBoost,1.000000,0.252181,0.719852,0.657344,4,Moderate-high fidelity
9,pima_diabetes,Pima Diabetes,Healthcare,decision_tree,Decision Tree,1.000000,0.181610,0.744666,0.642092,5,Moderate-high fidelity


,dataset_name,dataset_label,domain,model_id,model_name,shap_reconstruction_fidelity_score,lime_fidelity_score,shap_lime_agreement_score,fidelity_score,fidelity_rank,fidelity_band
0,compas,COMPAS,Criminal Justice,xgboost,XGBoost,1.000000,0.518743,0.435232,0.651325,1,Moderate-high fidelity
1,compas,COMPAS,Criminal Justice,random_forest,Random Forest,0.999444,0.404200,0.439104,0.614249,2,Moderate-high fidelity
2,compas,COMPAS,Criminal Justice,mlp,MLP,1.000000,0.456811,0.345068,0.600626,3,Moderate-high fidelity
3,compas,COMPAS,Criminal Justice,decision_tree,Decision Tree,1.000000,0.263802,0.511505,0.591769,4,Moderate fidelity
4,compas,COMPAS,Criminal Justice,logistic_regression,Logistic Regression,1.000000,0.366766,0.336659,0.567808,5,Moderate fidelity
5,german_credit,German Credit,Credit Risk,logistic_regression,Logistic Regression,1.000000,0.663835,0.536884,0.733573,1,Moderate-high fidelity
6,german_credit,German Credit,Credit Risk,random_forest,Random Forest,0.999333,0.514927,0.588673,0.700978,2,Moderate-high fidelity
7,german_credit,German Credit,Credit Risk,xgboost,XGBoost,1.000000,0.396174,0.551109,0.649094,3,Moderate-high fidelity
8,german_credit,German Credit,Credit Risk,decision_tree,Decision Tree,1.000000,0.333585,0.574753,0.636113,4,Moderate-high fidelity
9,german_credit,German Credit,Credit Risk,mlp,MLP,1.000000,0.171749,0.347755,0.506501,5,Moderate fidelity


# 11. Stability Pillar Comparison

## Objective

Compare stability under local perturbation.

## Storyline Example

The winner table identifies the strongest model per dataset, while the component breakdown explains why it won.

In [12]:
# =========================================================
# 11. Stability Pillar Comparison
# =========================================================

base_cols = ["dataset_name", "dataset_label", "domain", "model_id", "model_name"]
stability_component_df = stability_all_df[safe_cols(stability_all_df, base_cols + ['mean_local_lipschitz_ratio', 'stability_score', 'stability_rank', 'stability_band'])].copy()

if not stability_component_df.empty and "stability_rank" not in stability_component_df.columns and "stability_score" in stability_component_df.columns:
    stability_component_df = rank_within_dataset(stability_component_df, "stability_score", "stability_rank", ascending=False)

stability_winner_df = get_dataset_winners(stability_component_df, "stability_score", "stability_rank", "stability_score") if not stability_component_df.empty and "stability_score" in stability_component_df.columns else pd.DataFrame()

stability_ranking_df = (
    stability_component_df
    .sort_values(["dataset_name", "stability_rank", "model_id"])
    .reset_index(drop=True)
) if not stability_component_df.empty and "stability_rank" in stability_component_df.columns else pd.DataFrame()

display(stability_winner_df)
display(stability_component_df)
display(stability_ranking_df)

,dataset_name,dataset_label,domain,winner_model_id,winner_model_name,stability_score
0,compas,COMPAS,Criminal Justice,decision_tree,Decision Tree,1.000000
1,german_credit,German Credit,Credit Risk,logistic_regression,Logistic Regression,1.000000
2,pima_diabetes,Pima Diabetes,Healthcare,logistic_regression,Logistic Regression,1.000000


,dataset_name,dataset_label,domain,model_id,model_name,mean_local_lipschitz_ratio,stability_score,stability_rank,stability_band
0,german_credit,German Credit,Credit Risk,logistic_regression,Logistic Regression,0.050224,1.000000,1,High stability
1,german_credit,German Credit,Credit Risk,random_forest,Random Forest,0.060819,0.972958,2,High stability
2,german_credit,German Credit,Credit Risk,decision_tree,Decision Tree,0.063646,0.965742,3,High stability
3,german_credit,German Credit,Credit Risk,mlp,MLP,0.180897,0.666459,4,Moderate-high stability
4,german_credit,German Credit,Credit Risk,xgboost,XGBoost,0.441999,0.000000,5,Low stability
5,pima_diabetes,Pima Diabetes,Healthcare,logistic_regression,Logistic Regression,0.079345,1.000000,1,High stability
6,pima_diabetes,Pima Diabetes,Healthcare,mlp,MLP,0.087623,0.952920,2,High stability
7,pima_diabetes,Pima Diabetes,Healthcare,random_forest,Random Forest,0.160645,0.537596,3,Moderate stability
8,pima_diabetes,Pima Diabetes,Healthcare,decision_tree,Decision Tree,0.166242,0.505760,4,Moderate stability
9,pima_diabetes,Pima Diabetes,Healthcare,xgboost,XGBoost,0.255165,0.000000,5,Low stability


,dataset_name,dataset_label,domain,model_id,model_name,mean_local_lipschitz_ratio,stability_score,stability_rank,stability_band
0,compas,COMPAS,Criminal Justice,decision_tree,Decision Tree,0.031236,1.000000,1,High stability
1,compas,COMPAS,Criminal Justice,logistic_regression,Logistic Regression,0.132007,0.661564,2,Moderate-high stability
2,compas,COMPAS,Criminal Justice,mlp,MLP,0.140674,0.632455,3,Moderate-high stability
3,compas,COMPAS,Criminal Justice,xgboost,XGBoost,0.144497,0.619616,4,Moderate-high stability
4,compas,COMPAS,Criminal Justice,random_forest,Random Forest,0.328991,0.000000,5,Low stability
5,german_credit,German Credit,Credit Risk,logistic_regression,Logistic Regression,0.050224,1.000000,1,High stability
6,german_credit,German Credit,Credit Risk,random_forest,Random Forest,0.060819,0.972958,2,High stability
7,german_credit,German Credit,Credit Risk,decision_tree,Decision Tree,0.063646,0.965742,3,High stability
8,german_credit,German Credit,Credit Risk,mlp,MLP,0.180897,0.666459,4,Moderate-high stability
9,german_credit,German Credit,Credit Risk,xgboost,XGBoost,0.441999,0.000000,5,Low stability


# 12. Actionability Pillar Comparison

## Objective

Compare Actionability and its subcomponents.

## Storyline Example

The winner table identifies the strongest model per dataset, while the component breakdown explains why it won.

In [13]:
# =========================================================
# 12. Actionability Pillar Comparison
# =========================================================

base_cols = ["dataset_name", "dataset_label", "domain", "model_id", "model_name"]
actionability_component_df = actionability_all_df[safe_cols(actionability_all_df, base_cols + ['availability_score', 'proximity_score', 'sparsity_score', 'actionability_score', 'actionability_rank', 'actionability_band'])].copy()

if not actionability_component_df.empty and "actionability_rank" not in actionability_component_df.columns and "actionability_score" in actionability_component_df.columns:
    actionability_component_df = rank_within_dataset(actionability_component_df, "actionability_score", "actionability_rank", ascending=False)

actionability_winner_df = get_dataset_winners(actionability_component_df, "actionability_score", "actionability_rank", "actionability_score") if not actionability_component_df.empty and "actionability_score" in actionability_component_df.columns else pd.DataFrame()

actionability_ranking_df = (
    actionability_component_df
    .sort_values(["dataset_name", "actionability_rank", "model_id"])
    .reset_index(drop=True)
) if not actionability_component_df.empty and "actionability_rank" in actionability_component_df.columns else pd.DataFrame()

display(actionability_winner_df)
display(actionability_component_df)
display(actionability_ranking_df)

,dataset_name,dataset_label,domain,winner_model_id,winner_model_name,actionability_score
0,compas,COMPAS,Criminal Justice,decision_tree,Decision Tree,0.950537
1,german_credit,German Credit,Credit Risk,decision_tree,Decision Tree,0.940371
2,pima_diabetes,Pima Diabetes,Healthcare,decision_tree,Decision Tree,0.942250


,dataset_name,dataset_label,domain,model_id,model_name,availability_score,proximity_score,sparsity_score,actionability_score,actionability_rank,actionability_band
0,german_credit,German Credit,Credit Risk,decision_tree,Decision Tree,1.000000,0.974709,0.846405,0.940371,1,High actionability
1,german_credit,German Credit,Credit Risk,xgboost,XGBoost,1.000000,0.955465,0.859410,0.938292,2,High actionability
2,german_credit,German Credit,Credit Risk,random_forest,Random Forest,1.000000,0.963671,0.831909,0.931860,3,High actionability
3,german_credit,German Credit,Credit Risk,mlp,MLP,1.000000,0.946515,0.847222,0.931246,4,High actionability
4,german_credit,German Credit,Credit Risk,logistic_regression,Logistic Regression,1.000000,0.953052,0.817664,0.923572,5,High actionability
5,pima_diabetes,Pima Diabetes,Healthcare,decision_tree,Decision Tree,1.000000,0.955848,0.870902,0.942250,1,High actionability
6,pima_diabetes,Pima Diabetes,Healthcare,xgboost,XGBoost,1.000000,0.952764,0.862745,0.938503,2,High actionability
7,pima_diabetes,Pima Diabetes,Healthcare,random_forest,Random Forest,1.000000,0.952648,0.855114,0.935921,3,High actionability
8,pima_diabetes,Pima Diabetes,Healthcare,mlp,MLP,1.000000,0.945213,0.857955,0.934389,4,High actionability
9,pima_diabetes,Pima Diabetes,Healthcare,logistic_regression,Logistic Regression,1.000000,0.943100,0.847826,0.930309,5,High actionability


,dataset_name,dataset_label,domain,model_id,model_name,availability_score,proximity_score,sparsity_score,actionability_score,actionability_rank,actionability_band
0,compas,COMPAS,Criminal Justice,decision_tree,Decision Tree,1.000000,0.959375,0.892237,0.950537,1,High actionability
1,compas,COMPAS,Criminal Justice,logistic_regression,Logistic Regression,1.000000,0.958794,0.887626,0.948806,2,High actionability
2,compas,COMPAS,Criminal Justice,xgboost,XGBoost,1.000000,0.955618,0.882705,0.946108,3,High actionability
3,compas,COMPAS,Criminal Justice,mlp,MLP,1.000000,0.950526,0.881861,0.944129,4,High actionability
4,compas,COMPAS,Criminal Justice,random_forest,Random Forest,1.000000,0.956268,0.875014,0.943761,5,High actionability
5,german_credit,German Credit,Credit Risk,decision_tree,Decision Tree,1.000000,0.974709,0.846405,0.940371,1,High actionability
6,german_credit,German Credit,Credit Risk,xgboost,XGBoost,1.000000,0.955465,0.859410,0.938292,2,High actionability
7,german_credit,German Credit,Credit Risk,random_forest,Random Forest,1.000000,0.963671,0.831909,0.931860,3,High actionability
8,german_credit,German Credit,Credit Risk,mlp,MLP,1.000000,0.946515,0.847222,0.931246,4,High actionability
9,german_credit,German Credit,Credit Risk,logistic_regression,Logistic Regression,1.000000,0.953052,0.817664,0.923572,5,High actionability


# 13. Regulatory Alignment Comparison

## Objective

Compare governance-readiness evidence.

## Storyline Example

The winner table identifies the strongest model per dataset, while the component breakdown explains why it won.

In [14]:
# =========================================================
# 13. Regulatory Alignment Comparison
# =========================================================

base_cols = ["dataset_name", "dataset_label", "domain", "model_id", "model_name"]
regulatory_component_df = regulatory_all_df[safe_cols(regulatory_all_df, base_cols + ['transparency_score', 'explanation_availability_score', 'fidelity_alignment_score', 'stability_alignment_score', 'actionability_contestability_score', 'sensitive_feature_awareness_score', 'documentation_auditability_score', 'regulatory_alignment_score', 'regulatory_alignment_rank', 'regulatory_alignment_band'])].copy()

if not regulatory_component_df.empty and "regulatory_alignment_rank" not in regulatory_component_df.columns and "regulatory_alignment_score" in regulatory_component_df.columns:
    regulatory_component_df = rank_within_dataset(regulatory_component_df, "regulatory_alignment_score", "regulatory_alignment_rank", ascending=False)

regulatory_winner_df = get_dataset_winners(regulatory_component_df, "regulatory_alignment_score", "regulatory_alignment_rank", "regulatory_alignment_score") if not regulatory_component_df.empty and "regulatory_alignment_score" in regulatory_component_df.columns else pd.DataFrame()

regulatory_ranking_df = (
    regulatory_component_df
    .sort_values(["dataset_name", "regulatory_alignment_rank", "model_id"])
    .reset_index(drop=True)
) if not regulatory_component_df.empty and "regulatory_alignment_rank" in regulatory_component_df.columns else pd.DataFrame()

display(regulatory_winner_df)
display(regulatory_component_df)
display(regulatory_ranking_df)

,dataset_name,dataset_label,domain,winner_model_id,winner_model_name,regulatory_alignment_score
0,compas,COMPAS,Criminal Justice,decision_tree,Decision Tree,1.000000
1,compas,COMPAS,Criminal Justice,logistic_regression,Logistic Regression,1.000000
2,compas,COMPAS,Criminal Justice,mlp,MLP,1.000000
3,compas,COMPAS,Criminal Justice,random_forest,Random Forest,1.000000
4,compas,COMPAS,Criminal Justice,xgboost,XGBoost,1.000000
5,german_credit,German Credit,Credit Risk,decision_tree,Decision Tree,1.000000
6,german_credit,German Credit,Credit Risk,logistic_regression,Logistic Regression,1.000000
7,german_credit,German Credit,Credit Risk,mlp,MLP,1.000000
8,german_credit,German Credit,Credit Risk,random_forest,Random Forest,1.000000
9,german_credit,German Credit,Credit Risk,xgboost,XGBoost,1.000000


,dataset_name,dataset_label,domain,model_id,model_name,transparency_score,explanation_availability_score,sensitive_feature_awareness_score,documentation_auditability_score,regulatory_alignment_score,regulatory_alignment_rank
0,german_credit,German Credit,Credit Risk,decision_tree,Decision Tree,1.000000,1.000000,1.000000,1.000000,1.000000,1
1,german_credit,German Credit,Credit Risk,logistic_regression,Logistic Regression,1.000000,1.000000,1.000000,1.000000,1.000000,1
2,german_credit,German Credit,Credit Risk,mlp,MLP,1.000000,1.000000,1.000000,1.000000,1.000000,1
3,german_credit,German Credit,Credit Risk,random_forest,Random Forest,1.000000,1.000000,1.000000,1.000000,1.000000,1
4,german_credit,German Credit,Credit Risk,xgboost,XGBoost,1.000000,1.000000,1.000000,1.000000,1.000000,1
5,pima_diabetes,Pima Diabetes,Healthcare,decision_tree,Decision Tree,1.000000,1.000000,1.000000,1.000000,1.000000,1
6,pima_diabetes,Pima Diabetes,Healthcare,logistic_regression,Logistic Regression,1.000000,1.000000,1.000000,1.000000,1.000000,1
7,pima_diabetes,Pima Diabetes,Healthcare,mlp,MLP,1.000000,1.000000,1.000000,1.000000,1.000000,1
8,pima_diabetes,Pima Diabetes,Healthcare,random_forest,Random Forest,1.000000,1.000000,1.000000,1.000000,1.000000,1
9,pima_diabetes,Pima Diabetes,Healthcare,xgboost,XGBoost,1.000000,1.000000,1.000000,1.000000,1.000000,1


,dataset_name,dataset_label,domain,model_id,model_name,transparency_score,explanation_availability_score,sensitive_feature_awareness_score,documentation_auditability_score,regulatory_alignment_score,regulatory_alignment_rank
0,compas,COMPAS,Criminal Justice,decision_tree,Decision Tree,1.000000,1.000000,1.000000,1.000000,1.000000,1
1,compas,COMPAS,Criminal Justice,logistic_regression,Logistic Regression,1.000000,1.000000,1.000000,1.000000,1.000000,1
2,compas,COMPAS,Criminal Justice,mlp,MLP,1.000000,1.000000,1.000000,1.000000,1.000000,1
3,compas,COMPAS,Criminal Justice,random_forest,Random Forest,1.000000,1.000000,1.000000,1.000000,1.000000,1
4,compas,COMPAS,Criminal Justice,xgboost,XGBoost,1.000000,1.000000,1.000000,1.000000,1.000000,1
5,german_credit,German Credit,Credit Risk,decision_tree,Decision Tree,1.000000,1.000000,1.000000,1.000000,1.000000,1
6,german_credit,German Credit,Credit Risk,logistic_regression,Logistic Regression,1.000000,1.000000,1.000000,1.000000,1.000000,1
7,german_credit,German Credit,Credit Risk,mlp,MLP,1.000000,1.000000,1.000000,1.000000,1.000000,1
8,german_credit,German Credit,Credit Risk,random_forest,Random Forest,1.000000,1.000000,1.000000,1.000000,1.000000,1
9,german_credit,German Credit,Credit Risk,xgboost,XGBoost,1.000000,1.000000,1.000000,1.000000,1.000000,1


# 14. Final EES Comparison

## Objective

Compare final EES and its four pillars.

## Storyline Example

The winner table identifies the strongest model per dataset, while the component breakdown explains why it won.

In [15]:
# =========================================================
# 14. Final EES Comparison
# =========================================================

base_cols = ["dataset_name", "dataset_label", "domain", "model_id", "model_name"]
ees_component_df = ees_all_df[safe_cols(ees_all_df, base_cols + ['fidelity_score', 'stability_score', 'actionability_score', 'regulatory_alignment_score', 'ees_score', 'ees_score_pct', 'ees_rank', 'ees_band', 'scheme_1_equal_weight_score', 'scheme_6_theoretical_priority_score'])].copy()

if not ees_component_df.empty and "ees_rank" not in ees_component_df.columns and "ees_score" in ees_component_df.columns:
    ees_component_df = rank_within_dataset(ees_component_df, "ees_score", "ees_rank", ascending=False)

ees_winner_df = get_dataset_winners(ees_component_df, "ees_score", "ees_rank", "ees_score") if not ees_component_df.empty and "ees_score" in ees_component_df.columns else pd.DataFrame()

ees_ranking_df = (
    ees_component_df
    .sort_values(["dataset_name", "ees_rank", "model_id"])
    .reset_index(drop=True)
) if not ees_component_df.empty and "ees_rank" in ees_component_df.columns else pd.DataFrame()

display(ees_winner_df)
display(ees_component_df)
display(ees_ranking_df)

,dataset_name,dataset_label,domain,winner_model_id,winner_model_name,ees_score
0,compas,COMPAS,Criminal Justice,decision_tree,Decision Tree,0.885577
1,german_credit,German Credit,Credit Risk,logistic_regression,Logistic Regression,0.914286
2,pima_diabetes,Pima Diabetes,Healthcare,logistic_regression,Logistic Regression,0.914646


,dataset_name,dataset_label,domain,model_id,model_name,fidelity_score,stability_score,actionability_score,regulatory_alignment_score,ees_score,ees_score_pct,ees_rank,ees_band,scheme_1_equal_weight_score,scheme_6_theoretical_priority_score
0,german_credit,German Credit,Credit Risk,logistic_regression,Logistic Regression,0.733573,1.000000,0.923572,1.000000,0.914286,91.428618,1,High EES,0.914286,0.868206
1,german_credit,German Credit,Credit Risk,random_forest,Random Forest,0.700978,0.972958,0.931860,1.000000,0.901449,90.144887,2,High EES,0.901449,0.849436
2,german_credit,German Credit,Credit Risk,decision_tree,Decision Tree,0.636113,0.965742,0.940371,1.000000,0.885556,88.555646,3,High EES,0.885556,0.818782
3,german_credit,German Credit,Credit Risk,mlp,MLP,0.506501,0.666459,0.931246,1.000000,0.776052,77.605165,4,Moderate-high EES,0.776052,0.709432
4,german_credit,German Credit,Credit Risk,xgboost,XGBoost,0.649094,0.000000,0.938292,1.000000,0.646847,64.684652,5,Moderate-high EES,0.646847,0.670046
5,pima_diabetes,Pima Diabetes,Healthcare,logistic_regression,Logistic Regression,0.728274,1.000000,0.930309,1.000000,0.914646,91.464565,1,High EES,0.914646,0.866373
6,pima_diabetes,Pima Diabetes,Healthcare,mlp,MLP,0.694735,0.952920,0.934389,1.000000,0.895511,89.551103,2,High EES,0.895511,0.843552
7,pima_diabetes,Pima Diabetes,Healthcare,random_forest,Random Forest,0.680246,0.537596,0.935921,1.000000,0.788441,78.844059,3,Moderate-high EES,0.788441,0.770439
8,pima_diabetes,Pima Diabetes,Healthcare,decision_tree,Decision Tree,0.642092,0.505760,0.942250,1.000000,0.772525,77.252541,4,Moderate-high EES,0.772525,0.748117
9,pima_diabetes,Pima Diabetes,Healthcare,xgboost,XGBoost,0.657344,0.000000,0.938503,1.000000,0.648962,64.896182,5,Moderate-high EES,0.648962,0.673921


,dataset_name,dataset_label,domain,model_id,model_name,fidelity_score,stability_score,actionability_score,regulatory_alignment_score,ees_score,ees_score_pct,ees_rank,ees_band,scheme_1_equal_weight_score,scheme_6_theoretical_priority_score
0,compas,COMPAS,Criminal Justice,decision_tree,Decision Tree,0.591769,1.000000,0.950537,1.000000,0.885577,88.557657,1,High EES,0.885577,0.804515
1,compas,COMPAS,Criminal Justice,xgboost,XGBoost,0.651325,0.619616,0.946108,1.000000,0.804262,80.426211,2,High EES,0.804262,0.771027
2,compas,COMPAS,Criminal Justice,logistic_regression,Logistic Regression,0.567808,0.661564,0.948806,1.000000,0.794545,79.454478,3,Moderate-high EES,0.794545,0.738973
3,compas,COMPAS,Criminal Justice,mlp,MLP,0.600626,0.632455,0.944129,1.000000,0.794303,79.430268,4,Moderate-high EES,0.794303,0.749203
4,compas,COMPAS,Criminal Justice,random_forest,Random Forest,0.614249,0.000000,0.943761,1.000000,0.639503,63.950256,5,Moderate-high EES,0.639503,0.654285
5,german_credit,German Credit,Credit Risk,logistic_regression,Logistic Regression,0.733573,1.000000,0.923572,1.000000,0.914286,91.428618,1,High EES,0.914286,0.868206
6,german_credit,German Credit,Credit Risk,random_forest,Random Forest,0.700978,0.972958,0.931860,1.000000,0.901449,90.144887,2,High EES,0.901449,0.849436
7,german_credit,German Credit,Credit Risk,decision_tree,Decision Tree,0.636113,0.965742,0.940371,1.000000,0.885556,88.555646,3,High EES,0.885556,0.818782
8,german_credit,German Credit,Credit Risk,mlp,MLP,0.506501,0.666459,0.931246,1.000000,0.776052,77.605165,4,Moderate-high EES,0.776052,0.709432
9,german_credit,German Credit,Credit Risk,xgboost,XGBoost,0.649094,0.000000,0.938292,1.000000,0.646847,64.684652,5,Moderate-high EES,0.646847,0.670046


# 15. Cross-Dataset Average EES

## Objective

Identify which model family has the strongest average EES performance across domains.

In [16]:
# =========================================================
# 15. Cross-Dataset Average EES
# =========================================================

if ees_component_df.empty:
    cross_dataset_average_ees_df = pd.DataFrame()
else:
    cross_dataset_average_ees_df = (
        ees_component_df
        .groupby(["model_id", "model_name"], as_index=False)
        .agg(
            mean_ees_score=("ees_score", "mean"),
            median_ees_score=("ees_score", "median"),
            min_ees_score=("ees_score", "min"),
            max_ees_score=("ees_score", "max"),
            n_datasets=("dataset_name", "nunique")
        )
        .sort_values("mean_ees_score", ascending=False)
        .reset_index(drop=True)
    )
    cross_dataset_average_ees_df["cross_dataset_ees_rank"] = cross_dataset_average_ees_df["mean_ees_score"].rank(ascending=False, method="dense").astype(int)

display(cross_dataset_average_ees_df)

,model_id,model_name,mean_ees_score,median_ees_score,min_ees_score,max_ees_score,n_datasets,cross_dataset_ees_rank
0,logistic_regression,Logistic Regression,0.874492,0.914286,0.794545,0.914646,3,1
1,decision_tree,Decision Tree,0.847886,0.885556,0.772525,0.885577,3,2
2,mlp,MLP,0.821955,0.794303,0.776052,0.895511,3,3
3,random_forest,Random Forest,0.776464,0.788441,0.639503,0.901449,3,4
4,xgboost,XGBoost,0.700023,0.648962,0.646847,0.804262,3,5


# 16. AHP Sensitivity and Ranking Robustness

## Objective

Assess whether EES rankings remain stable under alternative AHP-derived weighting schemes.

In [17]:
# =========================================================
# 16. AHP Sensitivity and Ranking Robustness
# =========================================================

ahp_scheme_summary_df = ahp_scheme_weights_all_df.copy().sort_values(safe_cols(ahp_scheme_weights_all_df, ["dataset_name", "scheme_id", "pillar"])).reset_index(drop=True) if not ahp_scheme_weights_all_df.empty else pd.DataFrame()
ranking_robustness_comparison_df = ranking_robustness_all_df.copy().sort_values(safe_cols(ranking_robustness_all_df, ["dataset_name", "mean_rank_across_schemes", "model_id"])).reset_index(drop=True) if not ranking_robustness_all_df.empty else pd.DataFrame()
ahp_diagnostics_comparison_df = ahp_diagnostics_all_df.copy().sort_values(safe_cols(ahp_diagnostics_all_df, ["dataset_name", "scheme_id"])).reset_index(drop=True) if not ahp_diagnostics_all_df.empty else pd.DataFrame()

display(ahp_scheme_summary_df.head(40))
display(ranking_robustness_comparison_df)
display(ahp_diagnostics_comparison_df)

,dataset_name,dataset_label,domain,scheme_id,scheme_name,scheme_role,pillar,pillar_column,ahp_weight,ahp_weight_pct
0,compas,COMPAS,Criminal Justice,scheme_1_equal_weight,Scheme 1 - Equal Weight,Primary Master's EES score,Actionability,actionability_score,0.250000,25.000000
1,compas,COMPAS,Criminal Justice,scheme_1_equal_weight,Scheme 1 - Equal Weight,Primary Master's EES score,Fidelity,fidelity_score,0.250000,25.000000
2,compas,COMPAS,Criminal Justice,scheme_1_equal_weight,Scheme 1 - Equal Weight,Primary Master's EES score,Regulatory Alignment,regulatory_alignment_score,0.250000,25.000000
3,compas,COMPAS,Criminal Justice,scheme_1_equal_weight,Scheme 1 - Equal Weight,Primary Master's EES score,Stability,stability_score,0.250000,25.000000
4,compas,COMPAS,Criminal Justice,scheme_2_fidelity_priority,Scheme 2 - Fidelity Priority,Sensitivity analysis,Actionability,actionability_score,0.200000,20.000000
5,compas,COMPAS,Criminal Justice,scheme_2_fidelity_priority,Scheme 2 - Fidelity Priority,Sensitivity analysis,Fidelity,fidelity_score,0.400000,40.000000
6,compas,COMPAS,Criminal Justice,scheme_2_fidelity_priority,Scheme 2 - Fidelity Priority,Sensitivity analysis,Regulatory Alignment,regulatory_alignment_score,0.200000,20.000000
7,compas,COMPAS,Criminal Justice,scheme_2_fidelity_priority,Scheme 2 - Fidelity Priority,Sensitivity analysis,Stability,stability_score,0.200000,20.000000
8,compas,COMPAS,Criminal Justice,scheme_3_stability_priority,Scheme 3 - Stability Priority,Sensitivity analysis,Actionability,actionability_score,0.200000,20.000000
9,compas,COMPAS,Criminal Justice,scheme_3_stability_priority,Scheme 3 - Stability Priority,Sensitivity analysis,Fidelity,fidelity_score,0.200000,20.000000


,dataset_name,dataset_label,domain,model_id,model_name,mean_rank_across_schemes,median_rank_across_schemes,min_rank_across_schemes,max_rank_across_schemes,rank_range_across_schemes,mean_score_across_schemes,min_score_across_schemes,max_score_across_schemes,max_rank_change_vs_equal_weight,mean_rank_change_vs_equal_weight,ranking_robustness_note
0,compas,COMPAS,Criminal Justice,decision_tree,Decision Tree,1.000000,1.000000,1,1,0,0.872066,0.804515,0.908461,0,0.000000,No rank change across sensitivity schemes.
1,compas,COMPAS,Criminal Justice,xgboost,XGBoost,2.166667,2.000000,2,3,1,0.798723,0.767333,0.843410,1,0.166667,Minor rank movement across sensitivity schemes.
2,compas,COMPAS,Criminal Justice,logistic_regression,Logistic Regression,3.166667,3.000000,2,4,2,0.785283,0.738973,0.835636,1,0.500000,Minor rank movement across sensitivity schemes.
3,compas,COMPAS,Criminal Justice,mlp,MLP,3.666667,4.000000,3,4,1,0.786786,0.749203,0.835442,1,0.333333,Minor rank movement across sensitivity schemes.
4,compas,COMPAS,Criminal Justice,random_forest,Random Forest,5.000000,5.000000,5,5,0,0.641966,0.511602,0.711602,0,0.000000,No rank change across sensitivity schemes.
5,german_credit,German Credit,Credit Risk,logistic_regression,Logistic Regression,1.000000,1.000000,1,1,0,0.906606,0.868206,0.931429,0,0.000000,No rank change across sensitivity schemes.
6,german_credit,German Credit,Credit Risk,random_forest,Random Forest,2.000000,2.000000,2,2,0,0.892780,0.849436,0.921159,0,0.000000,No rank change across sensitivity schemes.
7,german_credit,German Credit,Credit Risk,decision_tree,Decision Tree,3.000000,3.000000,3,3,0,0.874427,0.818782,0.908445,0,0.000000,No rank change across sensitivity schemes.
8,german_credit,German Credit,Credit Risk,mlp,MLP,4.000000,4.000000,4,4,0,0.764948,0.709432,0.820841,0,0.000000,No rank change across sensitivity schemes.
9,german_credit,German Credit,Credit Risk,xgboost,XGBoost,5.000000,5.000000,5,5,0,0.650713,0.517477,0.717477,0,0.000000,No rank change across sensitivity schemes.


,dataset_name,dataset_label,domain,scheme_id,scheme_name,scheme_role,lambda_max,consistency_index,random_index,consistency_ratio,is_consistent_under_0_10
0,compas,COMPAS,Criminal Justice,scheme_1_equal_weight,Scheme 1 - Equal Weight,Primary Master's EES score,4.000000,-0.000000,0.900000,-0.000000,True
1,compas,COMPAS,Criminal Justice,scheme_2_fidelity_priority,Scheme 2 - Fidelity Priority,Sensitivity analysis,4.000000,-0.000000,0.900000,-0.000000,True
2,compas,COMPAS,Criminal Justice,scheme_3_stability_priority,Scheme 3 - Stability Priority,Sensitivity analysis,4.000000,-0.000000,0.900000,-0.000000,True
3,compas,COMPAS,Criminal Justice,scheme_4_actionability_priority,Scheme 4 - Actionability Priority,Sensitivity analysis,4.000000,-0.000000,0.900000,-0.000000,True
4,compas,COMPAS,Criminal Justice,scheme_5_regulatory_priority,Scheme 5 - Regulatory Alignment Priority,Sensitivity analysis,4.000000,-0.000000,0.900000,-0.000000,True
5,compas,COMPAS,Criminal Justice,scheme_6_theoretical_priority,Scheme 6 - Theoretical Priority F > R > S > A,Sensitivity analysis,4.030983,0.010328,0.900000,0.011475,True
6,german_credit,German Credit,Credit Risk,scheme_1_equal_weight,Scheme 1 - Equal Weight,Primary Master's EES score,4.000000,-0.000000,0.900000,-0.000000,True
7,german_credit,German Credit,Credit Risk,scheme_2_fidelity_priority,Scheme 2 - Fidelity Priority,Sensitivity analysis,4.000000,-0.000000,0.900000,-0.000000,True
8,german_credit,German Credit,Credit Risk,scheme_3_stability_priority,Scheme 3 - Stability Priority,Sensitivity analysis,4.000000,-0.000000,0.900000,-0.000000,True
9,german_credit,German Credit,Credit Risk,scheme_4_actionability_priority,Scheme 4 - Actionability Priority,Sensitivity analysis,4.000000,-0.000000,0.900000,-0.000000,True


# 17. Framework Validation Summary

## Objective

Summarise the cross-dataset findings into concise dissertation-ready outputs.

In [18]:
# =========================================================
# 17. Framework Validation Summary
# =========================================================

winner_tables = {
    "Business Score": baseline_winner_df,
    "LIME Fidelity": lime_winner_df,
    "SHAP-LIME Agreement": agreement_winner_df,
    "Counterfactual Actionability": counterfactual_winner_df,
    "Fidelity": fidelity_winner_df,
    "Stability": stability_winner_df,
    "Actionability": actionability_winner_df,
    "Regulatory Alignment": regulatory_winner_df,
    "Final EES": ees_winner_df
}

winner_matrix_rows = []
for section_name, winners_df in winner_tables.items():
    row = {"section": section_name}
    if not winners_df.empty:
        for _, winner_row in winners_df.iterrows():
            row[winner_row["dataset_label"]] = winner_row["winner_model_name"]
    winner_matrix_rows.append(row)

cross_dataset_winner_matrix_df = pd.DataFrame(winner_matrix_rows)

framework_validation_summary_rows = []
if not cross_dataset_average_ees_df.empty:
    top_overall = cross_dataset_average_ees_df.iloc[0]
    framework_validation_summary_rows.append({"finding_area": "Overall EES", "finding": f"{top_overall['model_name']} achieved the highest mean EES score across datasets."})
if not ranking_robustness_comparison_df.empty and "max_rank_change_vs_equal_weight" in ranking_robustness_comparison_df.columns:
    max_rank_change = ranking_robustness_comparison_df["max_rank_change_vs_equal_weight"].max()
    framework_validation_summary_rows.append({"finding_area": "AHP Sensitivity", "finding": f"The maximum observed rank change versus equal weighting was {max_rank_change}."})
framework_validation_summary_rows.append({"finding_area": "Framework Transparency", "finding": "Composite scores are reported with component breakdowns to preserve explainability of the EES Framework itself."})
framework_validation_summary_df = pd.DataFrame(framework_validation_summary_rows)

display(cross_dataset_winner_matrix_df)
display(framework_validation_summary_df)

,section,COMPAS,German Credit,Pima Diabetes
0,Business Score,Random Forest,XGBoost,Decision Tree
1,LIME Fidelity,XGBoost,Logistic Regression,Logistic Regression
2,SHAP-LIME Agreement,Decision Tree,Random Forest,Logistic Regression
3,Counterfactual Actionability,Decision Tree,Decision Tree,Decision Tree
4,Fidelity,XGBoost,Logistic Regression,Logistic Regression
5,Stability,Decision Tree,Logistic Regression,Logistic Regression
6,Actionability,Decision Tree,Decision Tree,Decision Tree
7,Regulatory Alignment,XGBoost,XGBoost,XGBoost
8,Final EES,Decision Tree,Logistic Regression,Logistic Regression


,finding_area,finding
0,Overall EES,Logistic Regression achieved the highest mean ...
1,AHP Sensitivity,The maximum observed rank change versus equal ...
2,Framework Transparency,Composite scores are reported with component b...


# 18. Publication-Ready Tables

## Objective

Create compact outputs suitable for dissertation and ESWA-style reporting.

In [19]:
# =========================================================
# 18. Publication-Ready Tables
# =========================================================

publication_table_1_dataset_overview_df = dataset_overview_df.copy()
publication_table_2_baseline_results_df = baseline_component_comparison_df[safe_cols(baseline_component_comparison_df, ["dataset_label", "domain", "model_name", "recall", "roc_auc", "precision", "business_priority_score", "business_priority_rank"])].copy()
publication_table_3_fidelity_results_df = fidelity_component_df[safe_cols(fidelity_component_df, ["dataset_label", "domain", "model_name", "shap_reconstruction_fidelity_score", "lime_fidelity_score", "shap_lime_agreement_score", "fidelity_score", "fidelity_rank"])].copy()
publication_table_4_actionability_results_df = actionability_component_df[safe_cols(actionability_component_df, ["dataset_label", "domain", "model_name", "availability_score", "proximity_score", "sparsity_score", "actionability_score", "actionability_rank"])].copy()
publication_table_5_ees_results_df = ees_component_df[safe_cols(ees_component_df, ["dataset_label", "domain", "model_name", "fidelity_score", "stability_score", "actionability_score", "regulatory_alignment_score", "ees_score", "ees_rank"])].copy()
publication_table_6_robustness_results_df = ranking_robustness_comparison_df[safe_cols(ranking_robustness_comparison_df, ["dataset_label", "domain", "model_name", "mean_rank_across_schemes", "min_rank_across_schemes", "max_rank_across_schemes", "rank_range_across_schemes", "max_rank_change_vs_equal_weight", "ranking_robustness_note"])].copy() if not ranking_robustness_comparison_df.empty else pd.DataFrame()

display(publication_table_1_dataset_overview_df)
display(publication_table_2_baseline_results_df.head(20))
display(publication_table_3_fidelity_results_df.head(20))
display(publication_table_4_actionability_results_df.head(20))
display(publication_table_5_ees_results_df.head(20))
display(publication_table_6_robustness_results_df.head(20))

,dataset_name,dataset_label,domain,pipeline_status,n_models
0,german_credit,German Credit,Credit Risk,Loaded,5
1,pima_diabetes,Pima Diabetes,Healthcare,Loaded,5
2,compas,COMPAS,Criminal Justice,Loaded,5


,dataset_label,domain,model_name,recall,roc_auc,precision,business_priority_score,business_priority_rank
0,German Credit,Credit Risk,XGBoost,0.516667,0.781429,0.632653,0.619293,1
1,German Credit,Credit Risk,Decision Tree,0.533333,0.739940,0.470588,0.582766,2
2,German Credit,Credit Risk,MLP,0.483333,0.740238,0.517857,0.567310,3
3,German Credit,Credit Risk,Random Forest,0.400000,0.769167,0.615385,0.553827,4
4,German Credit,Credit Risk,Logistic Regression,0.383333,0.761429,0.589744,0.538044,5
5,Pima Diabetes,Healthcare,Decision Tree,0.722222,0.762222,0.639344,0.717647,1
6,Pima Diabetes,Healthcare,XGBoost,0.648148,0.826481,0.686275,0.709273,2
7,Pima Diabetes,Healthcare,Random Forest,0.537037,0.815093,0.659091,0.644864,3
8,Pima Diabetes,Healthcare,MLP,0.500000,0.814630,0.613636,0.617116,4
9,Pima Diabetes,Healthcare,Logistic Regression,0.500000,0.812222,0.586957,0.611058,5


,dataset_label,domain,model_name,shap_reconstruction_fidelity_score,lime_fidelity_score,shap_lime_agreement_score,fidelity_score,fidelity_rank
0,German Credit,Credit Risk,Logistic Regression,1.000000,0.663835,0.536884,0.733573,1
1,German Credit,Credit Risk,Random Forest,0.999333,0.514927,0.588673,0.700978,2
2,German Credit,Credit Risk,XGBoost,1.000000,0.396174,0.551109,0.649094,3
3,German Credit,Credit Risk,Decision Tree,1.000000,0.333585,0.574753,0.636113,4
4,German Credit,Credit Risk,MLP,1.000000,0.171749,0.347755,0.506501,5
5,Pima Diabetes,Healthcare,Logistic Regression,1.000000,0.317475,0.867347,0.728274,1
6,Pima Diabetes,Healthcare,MLP,1.000000,0.290141,0.794063,0.694735,2
7,Pima Diabetes,Healthcare,Random Forest,0.998148,0.276355,0.766234,0.680246,3
8,Pima Diabetes,Healthcare,XGBoost,1.000000,0.252181,0.719852,0.657344,4
9,Pima Diabetes,Healthcare,Decision Tree,1.000000,0.181610,0.744666,0.642092,5


,dataset_label,domain,model_name,availability_score,proximity_score,sparsity_score,actionability_score,actionability_rank
0,German Credit,Credit Risk,Decision Tree,1.000000,0.974709,0.846405,0.940371,1
1,German Credit,Credit Risk,XGBoost,1.000000,0.955465,0.859410,0.938292,2
2,German Credit,Credit Risk,Random Forest,1.000000,0.963671,0.831909,0.931860,3
3,German Credit,Credit Risk,MLP,1.000000,0.946515,0.847222,0.931246,4
4,German Credit,Credit Risk,Logistic Regression,1.000000,0.953052,0.817664,0.923572,5
5,Pima Diabetes,Healthcare,Decision Tree,1.000000,0.955848,0.870902,0.942250,1
6,Pima Diabetes,Healthcare,XGBoost,1.000000,0.952764,0.862745,0.938503,2
7,Pima Diabetes,Healthcare,Random Forest,1.000000,0.952648,0.855114,0.935921,3
8,Pima Diabetes,Healthcare,MLP,1.000000,0.945213,0.857955,0.934389,4
9,Pima Diabetes,Healthcare,Logistic Regression,1.000000,0.943100,0.847826,0.930309,5


,dataset_label,domain,model_name,fidelity_score,stability_score,actionability_score,regulatory_alignment_score,ees_score,ees_rank
0,German Credit,Credit Risk,Logistic Regression,0.733573,1.000000,0.923572,1.000000,0.914286,1
1,German Credit,Credit Risk,Random Forest,0.700978,0.972958,0.931860,1.000000,0.901449,2
2,German Credit,Credit Risk,Decision Tree,0.636113,0.965742,0.940371,1.000000,0.885556,3
3,German Credit,Credit Risk,MLP,0.506501,0.666459,0.931246,1.000000,0.776052,4
4,German Credit,Credit Risk,XGBoost,0.649094,0.000000,0.938292,1.000000,0.646847,5
5,Pima Diabetes,Healthcare,Logistic Regression,0.728274,1.000000,0.930309,1.000000,0.914646,1
6,Pima Diabetes,Healthcare,MLP,0.694735,0.952920,0.934389,1.000000,0.895511,2
7,Pima Diabetes,Healthcare,Random Forest,0.680246,0.537596,0.935921,1.000000,0.788441,3
8,Pima Diabetes,Healthcare,Decision Tree,0.642092,0.505760,0.942250,1.000000,0.772525,4
9,Pima Diabetes,Healthcare,XGBoost,0.657344,0.000000,0.938503,1.000000,0.648962,5


,dataset_label,domain,model_name,mean_rank_across_schemes,min_rank_across_schemes,max_rank_across_schemes,rank_range_across_schemes,max_rank_change_vs_equal_weight,ranking_robustness_note
0,COMPAS,Criminal Justice,Decision Tree,1.000000,1,1,0,0,No rank change across sensitivity schemes.
1,COMPAS,Criminal Justice,XGBoost,2.166667,2,3,1,1,Minor rank movement across sensitivity schemes.
2,COMPAS,Criminal Justice,Logistic Regression,3.166667,2,4,2,1,Minor rank movement across sensitivity schemes.
3,COMPAS,Criminal Justice,MLP,3.666667,3,4,1,1,Minor rank movement across sensitivity schemes.
4,COMPAS,Criminal Justice,Random Forest,5.000000,5,5,0,0,No rank change across sensitivity schemes.
5,German Credit,Credit Risk,Logistic Regression,1.000000,1,1,0,0,No rank change across sensitivity schemes.
6,German Credit,Credit Risk,Random Forest,2.000000,2,2,0,0,No rank change across sensitivity schemes.
7,German Credit,Credit Risk,Decision Tree,3.000000,3,3,0,0,No rank change across sensitivity schemes.
8,German Credit,Credit Risk,MLP,4.000000,4,4,0,0,No rank change across sensitivity schemes.
9,German Credit,Credit Risk,XGBoost,5.000000,5,5,0,0,No rank change across sensitivity schemes.


# 19. Save Cross-Dataset Outputs

This section saves all cross-domain results to CSV and Excel.

In [20]:
# =========================================================
# 19. Save Cross-Dataset Outputs
# =========================================================

outputs = {
    "load_registry.csv": load_registry_df,
    "dataset_overview.csv": dataset_overview_df,
    "baseline_winner.csv": baseline_winner_df,
    "baseline_component_comparison.csv": baseline_component_comparison_df,
    "baseline_full_ranking.csv": baseline_full_ranking_df,
    "shap_top_feature_by_dataset.csv": shap_top_feature_by_dataset_df,
    "shap_top5_features_by_dataset.csv": shap_top5_features_by_dataset_df,
    "lime_winner.csv": lime_winner_df,
    "lime_ranking.csv": lime_ranking_df,
    "shap_lime_agreement_winner.csv": agreement_winner_df,
    "shap_lime_agreement_ranking.csv": agreement_ranking_df,
    "counterfactual_winner.csv": counterfactual_winner_df,
    "counterfactual_component_comparison.csv": counterfactual_component_df,
    "counterfactual_ranking.csv": counterfactual_ranking_df,
    "fidelity_winner.csv": fidelity_winner_df,
    "fidelity_component_comparison.csv": fidelity_component_df,
    "fidelity_ranking.csv": fidelity_ranking_df,
    "stability_winner.csv": stability_winner_df,
    "stability_component_comparison.csv": stability_component_df,
    "stability_ranking.csv": stability_ranking_df,
    "actionability_winner.csv": actionability_winner_df,
    "actionability_component_comparison.csv": actionability_component_df,
    "actionability_ranking.csv": actionability_ranking_df,
    "regulatory_winner.csv": regulatory_winner_df,
    "regulatory_component_comparison.csv": regulatory_component_df,
    "regulatory_ranking.csv": regulatory_ranking_df,
    "ees_winner.csv": ees_winner_df,
    "ees_component_comparison.csv": ees_component_df,
    "ees_ranking.csv": ees_ranking_df,
    "cross_dataset_average_ees.csv": cross_dataset_average_ees_df,
    "ahp_scheme_summary.csv": ahp_scheme_summary_df,
    "ranking_robustness_comparison.csv": ranking_robustness_comparison_df,
    "ahp_diagnostics_comparison.csv": ahp_diagnostics_comparison_df,
    "cross_dataset_winner_matrix.csv": cross_dataset_winner_matrix_df,
    "framework_validation_summary.csv": framework_validation_summary_df,
    "publication_table_1_dataset_overview.csv": publication_table_1_dataset_overview_df,
    "publication_table_2_baseline_results.csv": publication_table_2_baseline_results_df,
    "publication_table_3_fidelity_results.csv": publication_table_3_fidelity_results_df,
    "publication_table_4_actionability_results.csv": publication_table_4_actionability_results_df,
    "publication_table_5_ees_results.csv": publication_table_5_ees_results_df,
    "publication_table_6_robustness_results.csv": publication_table_6_robustness_results_df
}

for filename, df in outputs.items():
    save_df(df, filename)
    save_df(df, filename, folder=DATASETS_PATH)

save_excel(
    {
        "dataset_overview": dataset_overview_df,
        "baseline_components": baseline_component_comparison_df,
        "shap_top5": shap_top5_features_by_dataset_df,
        "lime_ranking": lime_ranking_df,
        "agreement_ranking": agreement_ranking_df,
        "counterfactual_components": counterfactual_component_df,
        "fidelity_components": fidelity_component_df,
        "stability_components": stability_component_df,
        "actionability_components": actionability_component_df,
        "regulatory_components": regulatory_component_df,
        "ees_components": ees_component_df,
        "ees_average": cross_dataset_average_ees_df,
        "ranking_robustness": ranking_robustness_comparison_df,
        "ahp_diagnostics": ahp_diagnostics_comparison_df,
        "winner_matrix": cross_dataset_winner_matrix_df,
        "validation_summary": framework_validation_summary_df,
        "load_registry": load_registry_df
    },
    "cross_dataset_framework_validation_outputs.xlsx"
)

manifest = {
    "notebook_name": "09_cross_dataset_framework_validation",
    "run_timestamp": datetime.now().isoformat(),
    "datasets": list(DATASET_CONFIGS.keys()),
    "random_state": RANDOM_STATE,
    "purpose": "Cross-domain validation of the EES Framework.",
    "top_shap_features": 5,
    "storyline_structure": ["Key Finding", "Winner Table", "Component Breakdown", "Full Ranking Table", "Cross-Dataset Interpretation"],
    "major_outputs": list(outputs.keys())
}

with open(LOGS_PATH / "cross_dataset_framework_validation_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)

with open(TABLES_PATH / "cross_dataset_framework_validation_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)

print("All cross-dataset outputs saved.")

Saved: /content/drive/MyDrive/Master_Research/07_Results/09_cross_dataset_framework_validation/tables/load_registry.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/09_cross_dataset_framework_validation/datasets/load_registry.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/09_cross_dataset_framework_validation/tables/dataset_overview.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/09_cross_dataset_framework_validation/datasets/dataset_overview.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/09_cross_dataset_framework_validation/tables/baseline_winner.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/09_cross_dataset_framework_validation/datasets/baseline_winner.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/09_cross_dataset_framework_validation/tables/baseline_component_comparison.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/09_cross_dataset_framework_validation/datasets/baseline_component_compa

# 20. Final Interpretation Note

This notebook is the cross-domain validation layer of the EES Framework.

The most important outputs for writing are:

1. `cross_dataset_winner_matrix.csv`
2. `cross_dataset_average_ees.csv`
3. `baseline_component_comparison.csv`
4. `fidelity_component_comparison.csv`
5. `actionability_component_comparison.csv`
6. `ees_component_comparison.csv`
7. `ranking_robustness_comparison.csv`
8. `cross_dataset_framework_validation_outputs.xlsx`

In [21]:
# =========================================================
# 20. Final Completion Message
# =========================================================

print("=" * 100)
print("09 Cross-Dataset Framework Validation completed successfully.")
print("=" * 100)

print(f"Tables saved to:   {TABLES_PATH}")
print(f"Datasets saved to: {DATASETS_PATH}")
print(f"Logs saved to:     {LOGS_PATH}")

print("\nKey outputs:")
print("- cross_dataset_framework_validation_outputs.xlsx")
print("- cross_dataset_winner_matrix.csv")
print("- cross_dataset_average_ees.csv")
print("- baseline_component_comparison.csv")
print("- fidelity_component_comparison.csv")
print("- actionability_component_comparison.csv")
print("- ees_component_comparison.csv")
print("- ranking_robustness_comparison.csv")

09 Cross-Dataset Framework Validation completed successfully.
Tables saved to:   /content/drive/MyDrive/Master_Research/07_Results/09_cross_dataset_framework_validation/tables
Datasets saved to: /content/drive/MyDrive/Master_Research/07_Results/09_cross_dataset_framework_validation/datasets
Logs saved to:     /content/drive/MyDrive/Master_Research/07_Results/09_cross_dataset_framework_validation/logs

Key outputs:
- cross_dataset_framework_validation_outputs.xlsx
- cross_dataset_winner_matrix.csv
- cross_dataset_average_ees.csv
- baseline_component_comparison.csv
- fidelity_component_comparison.csv
- actionability_component_comparison.csv
- ees_component_comparison.csv
- ranking_robustness_comparison.csv
